# Local RAG Pipeline with HuggingFace Embeddings (Python 3.12)

This notebook demonstrates how to:

- ~~Read local `.txt` and `.pdf` files~~
  - ~~This is good enough for 'playing' but production would need many more examples of files AND longer files.~~
  - ~~A single service would not be enough``~~
- ~~Chunk text for embedding~~
- ~~Generate embeddings using a local HuggingFace model~~
- ~~Save and load vector index locally~~
- Perform cosine similarity search without any external services

In [2]:
%pip install --upgrade pip

%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from config import settings
# from scikit-learn.metrics.pairwise import cosine_similarity
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np
import pandas as pd

sklearn.show_versions()


System:
    python: 3.12.6 (main, Sep  9 2024, 21:33:51) [Clang 18.1.8 ]
executable: /Users/davidclare/Projects/jupyter-notebooks/proj_rag/.venv/bin/python
   machine: macOS-15.3.2-x86_64-i386-64bit

Python dependencies:
      sklearn: 1.6.1
          pip: 25.0.1
   setuptools: None
        numpy: 1.26.4
        scipy: 1.15.2
       Cython: None
       pandas: 2.2.3
   matplotlib: None
       joblib: 1.4.2
threadpoolctl: 3.6.0

Built with OpenMP: True

threadpoolctl info:
       user_api: openmp
   internal_api: openmp
    num_threads: 12
         prefix: libomp
       filepath: /Users/davidclare/Projects/jupyter-notebooks/proj_rag/.venv/lib/python3.12/site-packages/sklearn/.dylibs/libomp.dylib
        version: None

       user_api: blas
   internal_api: openblas
    num_threads: 6
         prefix: libopenblas
       filepath: /Users/davidclare/Projects/jupyter-notebooks/proj_rag/.venv/lib/python3.12/site-packages/numpy/.dylibs/libopenblas64_.0.dylib
        version: 0.3.23.dev
threa

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [11]:
# Load from parquet
df = pd.read_parquet(settings.VECTOR_STORE)

print(df.head())

top_k = 3

# New query
query = "What is the critical spacing on holes"
query_emb = model.encode([query])[0]

                                                text  \
0       __META__FILE__NAME: citation-374847716.txt\n   
1     __ID__: 6794a1e3-abb5-43c4-84f6-81b9b581a3ec\n   
2                                 __META__CHUNK: 1\n   
3         __META__DATE: 2025-04-08T09:44:21.524045\n   
4  __CONTENT__: Robinet, Guibourt. (2006). Beobac...   

                                           embedding  
0  [-0.00135284464340657, 0.10549723356962204, -0...  
1  [-0.09487646073102951, 0.01802276074886322, -0...  
2  [-0.036805540323257446, 0.05023837834596634, -...  
3  [-0.0225250031799078, 0.025588540360331535, 0....  
4  [-0.03958938643336296, 0.032857973128557205, -...  


In [ ]:



# Similarity scores
scores = cosine_similarity([query_emb], df["embedding"].tolist())[0]

# Get top_k matches
top_indices = np.argsort(scores)[::-1][:top_k]


# Define a function to classify score
def classify_score(score):
    if score > 0.85:
        return "vhigh"
    elif score > 0.7:
        return "high"
    elif score > 0.55:
        return "med"
    elif score > 0.4:
        return "low"
    else:
        return "vlow"


# # Show top results
# for idx in top_indices:
#     print(f"Score: {scores[idx]:.4f}")
#     print(df.iloc[idx]["text"])
#     print("-" * 80)

# Show top results
results = []
for idx in top_indices:
    row = df.iloc[idx]
    score = scores[idx]
    results.append(
        {
            "score": round(score, 4),
            "level": classify_score(score),
            "id": row.get("__ID__", "N/A"),
            "meta": {
                "file": row.get("__META__FILE__NAME", ""),
                "chunk": row.get("__META__CHUNK", ""),
                "date": row.get("__META__DATE", ""),
            },
            "text": row["text"],
        }
    )

# Print or return results
for r in results:
    print(f"[{r['level']}] Score: {r['score']}, ID: {r['id']}")
    print(
        f"From file: {r['meta']['file']} (Chunk {r['meta']['chunk']}, Date: {r['meta']['date']})"
    )
    print(r["text"])
    print("-" * 80)

[low] Score: 0.4623, ID: N/A
From file:  (Chunk , Date: )
__CONTENT__: the holes to structures with precise but irregular shapes. The tech- form the desired surface. After the rods are in place, the nique is relatively inexpensive and adaptable to mass excess portion of the flat plates may be cut away. Theproduction. It was conceived originally for the fabrica- accuracy of the construction is assured through thetion of elliptical microwave reflectors, but may be positioning of the holes, and further precision fixtureuseful in constructing bridges and buildings and

--------------------------------------------------------------------------------
[low] Score: 0.4204, ID: N/A
From file:  (Chunk , Date: )
__CONTENT__: ough holes. Rout blank to size (1/ 16 in. ) per inch. Deburr edges - per inch. Clean and plate. Copper through holea (In. '). Nlckel-rhcdium tab ( in. '). Gold flash circult (~n.~). Connector tab and key slot Punch press both. Shear tab. Saw acd bevel slot. Connector tab - ch